# Agent Economics Framework - Guided Walkthrough

This notebook shows how the files in `src/deep_agents_foundry/economics/` connect, by walking the full pipeline one stage at a time: **raw economics runs -> metrics -> architecture summaries -> value frontier -> report -> saved JSON**.

Where the improvement framework (P8C) answers *"did this change make the agent better?"*, the economics framework (P8D) answers *"was that improvement worth the additional agentic work, cost, latency, and complexity?"*

It runs fully offline on small synthetic `EconomicsRun` records, so every cell returns instantly. No Azure, Foundry, web-search, or model calls are made here. In practice you would feed real per-run data in - for example by converting P8C comparison output with `economics_runs_from_comparison_report(...)`.

Two rules hold throughout: **missing metrics stay `None` (never coerced to zero)** and **token counts are never silently treated as dollars**.

## How the files connect

```mermaid
flowchart TD
    API["economics/__init__.py<br/>Public API"]
    MODELS["models.py<br/>EconomicsRun + result models"]
    METRICS["metrics.py<br/>Transparent cost/value formulas"]
    COMP["comparison.py<br/>Summaries, deltas, marginal returns"]
    FRONT["frontier.py<br/>Pareto / value frontier"]
    REPORT["reporting.py<br/>analyze + render report + routing"]
    LOG["economics_log.py<br/>Compact JSON persistence"]
    ADAPT["adapters.py<br/>Convert P8C outputs"]

    ADAPT --> MODELS
    API --> MODELS
    API --> METRICS
    API --> COMP
    API --> FRONT
    API --> REPORT
    API --> LOG
    COMP --> METRICS
    COMP --> MODELS
    FRONT --> MODELS
    REPORT --> COMP
    REPORT --> FRONT
    LOG --> REPORT
```

The pipeline reads top to bottom:

- `models.py` defines `EconomicsRun` (one run described in economic terms) plus the typed result records.
- `metrics.py` holds small, explicit formulas (effective cost, expected net value, cost per successful task, ...).
- `comparison.py` aggregates repeated runs into per-architecture summaries, computes interpretable deltas, and analyzes marginal returns.
- `frontier.py` computes the Pareto/value frontier and flags dominated architectures.
- `reporting.py` orchestrates the analysis, derives conservative routing observations, and renders the report.
- `economics_log.py` explicitly saves a compact JSON record under `artifacts/economics/`.
- `adapters.py` converts P8C outputs into `EconomicsRun` records - the analysis is downstream of P8C.

## Cell 1 - Imports

Import the public API from `deep_agents_foundry.economics`. Importing the package does not call Azure; it only wires the modules together. The routing states `PREFER`, `AVOID`, and `INVESTIGATE` are the only verdict vocabulary - there is deliberately no single combined "economics score".

In [1]:
import json
from collections import Counter
from pathlib import Path

from deep_agents_foundry.economics import (
    EconomicsRun,
    EconomicsRecord,
    summarize_architecture,
    summarize_architectures,
    compare_architectures,
    compute_value_frontier,
    analyze_marginal_returns,
    analyze_agent_economics,
    render_economics_report,
    save_economics_analysis,
    load_economics_record,
    list_economics_records,
    PREFER,
    AVOID,
    INVESTIGATE,
)

print("Imports OK")
print("Routing vocabulary:", PREFER, AVOID, INVESTIGATE)

Imports OK
Routing vocabulary: PREFER AVOID INVESTIGATE


## Cell 2 - Create economics run data

An `EconomicsRun` is one agent run described in economic terms: identity (`task_id`, `architecture`, `complexity`), token/work counts, latency, quality, task success, and optional cost / human-effort / business-value fields.

Here we synthesize a repeated experiment across **four architectures** x **three complexities**, with several repetitions each, so the aggregates are meaningful. The numbers are chosen to illustrate three realistic patterns:

- **simple** - deeper orchestration adds cost without improving quality (so it is dominated),
- **medium** - skills cut wasted searching, raising quality *and* lowering cost,
- **complex** - quality keeps climbing with cost, so no single architecture dominates.

> In a real study you would instead build these from live/offline runs, e.g. `economics_runs_from_comparison_report(p8c_report)` or `economics_runs_from_variant_result(variant, architecture=...)`.

In [2]:
def make_group(architecture, complexity, *, quality, agent_cost, total_tokens,
               web_searches, model_calls, subagent_calls, latency, successes):
    """Create repeated EconomicsRun records for one architecture x complexity."""
    runs = []
    for index, ok in enumerate(successes, start=1):
        runs.append(
            EconomicsRun(
                task_id=f"{complexity}-{index}",
                architecture=architecture,
                complexity=complexity,
                run_id=f"{architecture}-{complexity}-r{index}",
                input_tokens=int(total_tokens * 0.7),
                output_tokens=int(total_tokens * 0.3),
                total_tokens=total_tokens,
                model_calls=model_calls,
                web_searches=web_searches,
                tool_calls=web_searches,
                subagent_calls=subagent_calls,
                latency_seconds=latency,
                quality_score=quality,
                task_success=ok,
                agent_cost=agent_cost,
            )
        )
    return runs


ALL_OK = [True, True, True, True]
THREE_OK = [True, True, True, False]
HALF_OK = [True, True, False, False]

economics_runs = []

# SIMPLE: a light lookup where extra orchestration adds cost but not quality.
economics_runs += make_group("baseline-search", "simple", quality=4.4, agent_cost=0.05,
                             total_tokens=12000, web_searches=1, model_calls=2,
                             subagent_calls=0, latency=10, successes=ALL_OK)
economics_runs += make_group("deep-agent", "simple", quality=4.4, agent_cost=0.12,
                             total_tokens=20000, web_searches=2, model_calls=4,
                             subagent_calls=0, latency=18, successes=ALL_OK)
economics_runs += make_group("deep-agent-skills", "simple", quality=4.4, agent_cost=0.15,
                             total_tokens=21000, web_searches=2, model_calls=4,
                             subagent_calls=0, latency=19, successes=ALL_OK)
economics_runs += make_group("deep-agent-subagents", "simple", quality=4.3, agent_cost=0.30,
                             total_tokens=34000, web_searches=3, model_calls=6,
                             subagent_calls=2, latency=28, successes=ALL_OK)

# MEDIUM: skills cut wasted searching, improving quality while lowering cost.
economics_runs += make_group("baseline-search", "medium", quality=3.8, agent_cost=0.15,
                             total_tokens=15000, web_searches=2, model_calls=3,
                             subagent_calls=0, latency=16, successes=THREE_OK)
economics_runs += make_group("deep-agent", "medium", quality=4.1, agent_cost=0.14,
                             total_tokens=22000, web_searches=3, model_calls=5,
                             subagent_calls=0, latency=22, successes=ALL_OK)
economics_runs += make_group("deep-agent-skills", "medium", quality=4.6, agent_cost=0.12,
                             total_tokens=20000, web_searches=3, model_calls=5,
                             subagent_calls=0, latency=21, successes=ALL_OK)
economics_runs += make_group("deep-agent-subagents", "medium", quality=4.5, agent_cost=0.30,
                             total_tokens=33000, web_searches=4, model_calls=7,
                             subagent_calls=2, latency=30, successes=ALL_OK)

# COMPLEX: quality keeps rising with cost, so no single option dominates.
economics_runs += make_group("baseline-search", "complex", quality=3.5, agent_cost=0.20,
                             total_tokens=18000, web_searches=3, model_calls=4,
                             subagent_calls=0, latency=20, successes=HALF_OK)
economics_runs += make_group("deep-agent", "complex", quality=4.0, agent_cost=0.35,
                             total_tokens=30000, web_searches=5, model_calls=7,
                             subagent_calls=0, latency=32, successes=THREE_OK)
economics_runs += make_group("deep-agent-skills", "complex", quality=4.3, agent_cost=0.42,
                             total_tokens=34000, web_searches=6, model_calls=8,
                             subagent_calls=0, latency=36, successes=THREE_OK)
economics_runs += make_group("deep-agent-subagents", "complex", quality=4.8, agent_cost=0.80,
                             total_tokens=52000, web_searches=8, model_calls=12,
                             subagent_calls=3, latency=55, successes=ALL_OK)

print(f"Built {len(economics_runs)} synthetic economics runs")

Built 48 synthetic economics runs


## Cell 3 - Inspect the raw runs

Before any aggregation, look at what we actually have: how many runs per architecture/complexity group, and the full economic shape of a single run. `EconomicsRun.to_dict()` also exposes `resolved_total_tokens()`, which derives the total from input + output only when both are present (otherwise it stays `None`).

In [3]:
print("Total runs:", len(economics_runs))
print("Architectures:", sorted({run.architecture for run in economics_runs}))
print("Complexities:", sorted({run.complexity for run in economics_runs}))
print()

by_group = Counter((run.architecture, run.complexity) for run in economics_runs)
for (arch, complexity), count in sorted(by_group.items()):
    print(f"  {arch:24} {complexity:8} x{count}")

print()
print("Sample run (baseline-search / simple):")
for key, value in economics_runs[0].to_dict().items():
    print(f"  {key}: {value}")

Total runs: 48
Architectures: ['baseline-search', 'deep-agent', 'deep-agent-skills', 'deep-agent-subagents']
Complexities: ['complex', 'medium', 'simple']

  baseline-search          complex  x4
  baseline-search          medium   x4
  baseline-search          simple   x4
  deep-agent               complex  x4
  deep-agent               medium   x4
  deep-agent               simple   x4
  deep-agent-skills        complex  x4
  deep-agent-skills        medium   x4
  deep-agent-skills        simple   x4
  deep-agent-subagents     complex  x4
  deep-agent-subagents     medium   x4
  deep-agent-subagents     simple   x4

Sample run (baseline-search / simple):
  task_id: simple-1
  architecture: baseline-search
  complexity: simple
  run_id: baseline-search-simple-r1
  input_tokens: 8400
  output_tokens: 3600
  total_tokens: 12000
  resolved_total_tokens: 12000
  model_calls: 2
  web_searches: 1
  tool_calls: 1
  subagent_calls: 0
  memory_reads: None
  skill_loads: None
  summarization_cal

## Cell 4 - Summarize architectures (comparison.py)

`summarize_architectures` groups runs by `(architecture, complexity)` and averages each metric **only over runs that reported it**, so an unavailable metric stays `None` rather than being dragged toward zero. Each `ArchitectureSummary` also carries derived economics like `cost_per_successful_task` and `avg_effective_task_cost`.

Note the analysis stays separated by complexity - global averages would hide the fact that the right architecture depends on task difficulty.

In [4]:
summaries = summarize_architectures(economics_runs, by_complexity=True)


def fmt(value, spec):
    return "n/a" if value is None else format(value, spec)


header = (
    f"{'architecture':22} {'cmplx':8} {'runs':>4} {'succ':>5} {'qual':>5} "
    f"{'tokens':>7} {'cost':>6} {'cost/ok':>8}"
)
print(header)
print("-" * len(header))
for s in summaries:
    succ = "n/a" if s.success_rate is None else f"{s.success_rate * 100:.0f}%"
    print(
        f"{s.architecture:22} {str(s.complexity):8} {s.num_runs:>4} {succ:>5} "
        f"{fmt(s.avg_quality, '.2f'):>5} {fmt(s.avg_total_tokens, '.0f'):>7} "
        f"{fmt(s.avg_agent_cost, '.2f'):>6} {fmt(s.cost_per_successful_task, '.3f'):>8}"
    )

architecture           cmplx    runs  succ  qual  tokens   cost  cost/ok
------------------------------------------------------------------------
baseline-search        complex     4   50%  3.50   18000   0.20    0.400
baseline-search        medium      4   75%  3.80   15000   0.15    0.200
baseline-search        simple      4  100%  4.40   12000   0.05    0.050
deep-agent             complex     4   75%  4.00   30000   0.35    0.467
deep-agent             medium      4  100%  4.10   22000   0.14    0.140
deep-agent             simple      4  100%  4.40   20000   0.12    0.120
deep-agent-skills      complex     4   75%  4.30   34000   0.42    0.560
deep-agent-skills      medium      4  100%  4.60   20000   0.12    0.120
deep-agent-skills      simple      4  100%  4.40   21000   0.15    0.150
deep-agent-subagents   complex     4  100%  4.80   52000   0.80    0.800
deep-agent-subagents   medium      4  100%  4.50   33000   0.30    0.300
deep-agent-subagents   simple      4  100%  4.30   

## Cell 5 - Compare two architectures (comparison.py)

`compare_architectures` produces explicit `candidate - baseline` deltas across quality, success, work, and cost - and nothing else. There is no combined score; each metric stays interpretable and any unavailable side is reported as `None`. Here we contrast the cheapest and the deepest architecture on **simple** tasks to see whether the extra work paid off.

In [5]:
def summary_for(architecture, complexity):
    return summarize_architecture(
        economics_runs, architecture=architecture, complexity=complexity
    )


baseline_simple = summary_for("baseline-search", "simple")
subagents_simple = summary_for("deep-agent-subagents", "simple")

comparison = compare_architectures(baseline_simple, subagents_simple)
print(f"{comparison.candidate.architecture} vs {comparison.baseline.architecture} (simple)")
print()
for metric, delta in comparison.deltas.items():
    shown = "unavailable" if delta is None else f"{delta:+.4f}"
    print(f"  delta {metric:22} {shown}")
print()
for note in comparison.notes:
    print(" -", note)

deep-agent-subagents vs baseline-search (simple)

  delta quality                -0.1000
  delta success_rate           +0.0000
  delta total_tokens           +22000.0000
  delta latency_seconds        +18.0000
  delta web_searches           +2.0000
  delta model_calls            +4.0000
  delta tool_calls             +2.0000
  delta subagent_calls         +2.0000
  delta agent_cost             +0.2500
  delta effective_task_cost    +0.2500
  delta expected_net_value     unavailable

 - Quality lower by -0.10.
 - Success rate delta +0.0%.
 - 22000 more average tokens.
 - Effective task cost higher by +0.2500.
 - deep-agent-subagents vs baseline-search: metrics reported separately (no combined score).


## Cell 6 - Compute the value frontier (frontier.py)

`compute_value_frontier` maximizes a quality dimension while minimizing a cost dimension. An architecture is **dominated** when another has quality that is at-least-as-good *and* cost that is at-least-as-low, with at least one strict improvement.

We run it per complexity using `avg_agent_cost` as the cost dimension. Watch how the frontier changes with task difficulty:

- **simple** -> only `baseline-search` survives (the rest cost more for equal/lower quality),
- **medium** -> `deep-agent-skills` dominates (best quality *and* lowest cost),
- **complex** -> several architectures share the frontier (a genuine quality/cost trade-off).

If the cost dimension were unavailable, the result would say so explicitly instead of treating tokens as dollars.

In [6]:
for complexity in ("simple", "medium", "complex"):
    comp_summaries = [s for s in summaries if s.complexity == complexity]
    frontier = compute_value_frontier(
        comp_summaries,
        quality_dimension="avg_quality",
        cost_dimension="avg_agent_cost",
    )
    print(f"{complexity.upper()}  [{frontier.quality_dimension} vs {frontier.cost_dimension}]")
    if not frontier.available:
        print("  frontier unavailable:", frontier.reason)
        print()
        continue
    print("  frontier :", frontier.frontier)
    print("  dominated:", frontier.dominated)
    for entry in frontier.entries:
        if entry.dominated_by:
            print(f"    {entry.architecture} dominated by {entry.dominated_by}")
    print()

SIMPLE  [avg_quality vs avg_agent_cost]
  frontier : ['baseline-search']
  dominated: ['deep-agent', 'deep-agent-skills', 'deep-agent-subagents']
    deep-agent dominated by ['baseline-search']
    deep-agent-skills dominated by ['baseline-search', 'deep-agent']
    deep-agent-subagents dominated by ['baseline-search', 'deep-agent', 'deep-agent-skills']

MEDIUM  [avg_quality vs avg_agent_cost]
  frontier : ['deep-agent-skills']
  dominated: ['baseline-search', 'deep-agent', 'deep-agent-subagents']
    baseline-search dominated by ['deep-agent', 'deep-agent-skills']
    deep-agent dominated by ['deep-agent-skills']
    deep-agent-subagents dominated by ['deep-agent-skills']

COMPLEX  [avg_quality vs avg_agent_cost]
  frontier : ['baseline-search', 'deep-agent', 'deep-agent-skills', 'deep-agent-subagents']
  dominated: []



## Cell 7 - Analyze marginal returns (comparison.py)

`analyze_marginal_returns` buckets runs by a work metric (here `web_searches`) and reports how mean quality changes as work increases. It flags buckets where extra work yields little quality gain and names a **candidate elbow**.

Because marginal returns only make sense for the *same* kind of task at increasing work levels, we use a small dedicated search-depth study rather than the mixed dataset. The language stays deliberately tentative - this is *observed* evidence, not a statistical claim.

In [7]:
# Same task type, increasing search depth, with classic diminishing returns.
search_depth_runs = [
    EconomicsRun(
        task_id="depth-study",
        architecture="deep-agent-skills",
        complexity="medium",
        run_id=f"depth-{searches}",
        web_searches=searches,
        quality_score=quality,
    )
    for searches, quality in [(1, 4.0), (2, 4.6), (3, 4.8), (4, 4.81)]
]

marginal = analyze_marginal_returns(search_depth_runs, work_metric="web_searches")
print("Available:", marginal.available)
print("Candidate elbow:", marginal.candidate_elbow)
print()
print(f"{'searches':>8} {'quality':>8} {'d_work':>7} {'d_quality':>10} {'diminishing':>12}")
for point in marginal.points:
    dw = "-" if point.work_delta is None else f"{point.work_delta:+.0f}"
    dq = "-" if point.quality_delta is None else f"{point.quality_delta:+.2f}"
    print(f"{point.work_value:>8.0f} {point.quality_value:>8.2f} {dw:>7} {dq:>10} {str(point.diminishing):>12}")
print()
for obs in marginal.observations:
    print(" -", obs)

Available: True
Candidate elbow: 3.0

searches  quality  d_work  d_quality  diminishing
       1     4.00       -          -        False
       2     4.60      +1      +0.60        False
       3     4.80      +1      +0.20        False
       4     4.81      +1      +0.01         True

 - Additional web_searches beyond ~3 show little observed quality gain (suggests diminishing returns; not statistically certain).


## Cell 8 - Run the full economics analysis (reporting.py)

`analyze_agent_economics` ties everything together into one `EconomicsReport`: per-complexity summaries, value frontiers, marginal returns, and conservative **routing recommendations**. It also records which classes of metric were available (cost / human / business), so the report can distinguish *unavailable* from a genuine zero.

Routing stays conservative: it will `PREFER` a clear winner and fall back to `INVESTIGATE` when the frontier shows a real trade-off.

In [8]:
report = analyze_agent_economics(economics_runs, work_metric="web_searches")

print("Total runs:", report.total_runs)
print("Architectures:", report.architectures)
print("Complexities:", report.complexities)
print("Cost available:", report.cost_available)
print("Human economics available:", report.human_economics_available)
print("Business value available:", report.business_value_available)
print()
print("Routing:")
for rec in report.routing:
    target = rec.architecture or "(no single winner)"
    print(f"  {rec.complexity:8} -> {rec.state:12} {target}")
    print(f"           {rec.rationale}")

Total runs: 48
Architectures: ['baseline-search', 'deep-agent', 'deep-agent-skills', 'deep-agent-subagents']
Complexities: ['simple', 'medium', 'complex']
Cost available: True
Human economics available: False
Business value available: False

Routing:
  simple   -> PREFER       baseline-search
           baseline-search is the sole non-dominated architecture on the avg_quality/avg_agent_cost frontier.
  medium   -> PREFER       deep-agent-skills
           deep-agent-skills is the sole non-dominated architecture on the avg_quality/avg_agent_cost frontier.
  complex  -> INVESTIGATE  (no single winner)
           Frontier shows a genuine quality/cost trade-off (quality spread 1.30 on avg_quality); investigate before committing.


## Cell 9 - Render the human-readable report (reporting.py)

`render_economics_report` turns the `EconomicsReport` into a plain-text report with sections for data coverage, per-complexity tasks, value frontier, dominated architectures, marginal returns, human economics, business value, and recommended routing. Missing metrics render as `unavailable`; genuine zeros render as numbers.

In [9]:
print(render_economics_report(report))

Agent Economics Report

DATA COVERAGE
- 48 runs
- 4 architectures: baseline-search, deep-agent, deep-agent-skills, deep-agent-subagents
- Complexities: simple, medium, complex
- Direct dollar cost available
- Human economics unavailable
- Business value unavailable

SIMPLE TASKS
Preferred architecture: baseline-search
  baseline-search (runs: 4)
    Quality:        4.40
    Success:        100%
    Avg tokens:     12000
    Avg searches:   1.0
    Avg model calls:2.0
    Avg subagents:  0.0
    Avg latency:    10.0s
    Avg agent cost: 0.0500
    Effective cost: 0.0500
    Cost/success:   0.0500
  deep-agent (runs: 4)
    Quality:        4.40
    Success:        100%
    Avg tokens:     20000
    Avg searches:   2.0
    Avg model calls:4.0
    Avg subagents:  0.0
    Avg latency:    18.0s
    Avg agent cost: 0.1200
    Effective cost: 0.1200
    Cost/success:   0.1200
  deep-agent-skills (runs: 4)
    Quality:        4.40
    Success:        100%
    Avg tokens:     21000
    Avg searc

## Cell 10 - Save a compact economics record (economics_log.py)

Persistence is always **explicit** - `analyze_agent_economics` and `render_economics_report` never write files. `save_economics_analysis` converts the report into a compact `EconomicsRecord` (primitive values only; no raw traces or runtime objects) and writes one indented UTF-8 JSON file per analysis under `artifacts/economics/`.

We resolve the repo root so the file lands under the workspace `artifacts/economics/` regardless of the notebook's working directory.

In [10]:
def repo_root():
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return here


economics_dir = repo_root() / "artifacts" / "economics"
saved_path = save_economics_analysis(
    report,
    analysis_id="architecture-study-01",
    notes="Synthetic four-architecture study across simple/medium/complex.",
    directory=economics_dir,
)
print("Saved:", saved_path)

Saved: c:\Users\shchitt\Downloads\Projects\deep-agents-on-foundry\artifacts\economics\2026-09-13T181345Z_architecture-study-01.json


## Cell 11 - Inspect the saved record (economics_log.py)

Finally, reload what we persisted. `load_economics_record` reads one file back into an `EconomicsRecord`, and `list_economics_records` returns every saved analysis in chronological (filename) order. The record survives process exit and is ready for later review or trend analysis.

In [11]:
loaded = load_economics_record(saved_path)
print("analysis_id:", loaded.analysis_id)
print("timestamp:", loaded.timestamp)
print("preferred_by_complexity:", loaded.preferred_by_complexity)
print("frontier_by_complexity:", loaded.frontier_by_complexity)
print("dominated_by_complexity:", loaded.dominated_by_complexity)
print()
print("Records on disk:", [r.analysis_id for r in list_economics_records(economics_dir)])
print()
print("--- first 1000 chars of the saved JSON ---")
print(saved_path.read_text(encoding="utf-8")[:1000])

analysis_id: architecture-study-01
timestamp: 2026-09-13T18:13:45.161798+00:00
preferred_by_complexity: {'simple': 'baseline-search', 'medium': 'deep-agent-skills'}
frontier_by_complexity: {'simple': ['baseline-search'], 'medium': ['deep-agent-skills'], 'complex': ['baseline-search', 'deep-agent', 'deep-agent-skills', 'deep-agent-subagents']}
dominated_by_complexity: {'simple': ['deep-agent', 'deep-agent-skills', 'deep-agent-subagents'], 'medium': ['baseline-search', 'deep-agent', 'deep-agent-subagents'], 'complex': []}

Records on disk: ['architecture-study-01']

--- first 1000 chars of the saved JSON ---
{
  "analysis_id": "architecture-study-01",
  "total_runs": 48,
  "architectures": [
    "baseline-search",
    "deep-agent",
    "deep-agent-skills",
    "deep-agent-subagents"
  ],
  "complexities": [
    "simple",
    "medium",
    "complex"
  ],
  "cost_available": true,
  "human_economics_available": false,
  "business_value_available": false,
  "preferred_by_complexity": {
    